
# レジームスイッチングと多変量正規分布を用いた市場動態のモデリング

このノートブックでは、S&P 500 (SPX) と米国10年国債利回り (DGS10) のデータに対し、**隠れマルコフモデル的なアプローチ（GMMによるレジーム分類＋推移確率行列）**と**レジーム別の多変量正規分布（静的相関）**を組み合わせた分析パイプラインを解説します。

## 全体のパイプライン概要
1. **特徴量エンジニアリング**: SPXとDGS10の移動平均、標準偏差、相関係数を計算し、市場の「状態」を捉えやすくする。
2. **GMMによるクラスタリング**: 特徴量を元に、市場を4つの状態（レジーム）に分類する。
3. **推移確率行列の計算**: 状態間の遷移確率を経験的に計算する。
4. **状態ごとのARIMAモデリング（期待値の計算）**: 各状態でARIMAXモデルを推定し、条件付き期待値を算出する。
5. **レジーム別共分散行列の計算とシミュレーション**: 各レジームのARIMA予測残差から分散共分散行列（相関）を計算し、多変量正規分布からショックを発生させてシミュレーションを行う。


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from statsmodels.tsa.arima.model import ARIMA
import warnings

warnings.filterwarnings('ignore')



## 1. データの読み込みと特徴量エンジニアリング

### 理論
金融市場は常に同じ法則で動いているわけではなく、「ボラティリティが高い時期」「金利と株価が逆相関になる時期」など、マクロ的な環境（レジーム）が変化します。
この環境変化を捉えるため、単純な価格だけでなく、「20日（約1ヶ月）の移動平均、標準偏差、および相関」という**ローリング特徴量**を作成します。
これにより、瞬間的なノイズではなく、一定期間のトレンドやボラティリティの強さを元に状態を分類できるようになります。


In [ ]:
df = pd.read_csv('/home/u00118/train_sp500_us10y.csv')
if 'Unnamed: 0' in df.columns:
    df['Date'] = pd.to_datetime(df['Unnamed: 0'])
    df.set_index('Date', inplace=True)
    df.drop(columns=['Unnamed: 0'], inplace=True)

# ローリング特徴量の作成 (20日 = 約1ヶ月)
window = 20
df['sp500_roll_mean'] = df['sp500'].rolling(window=window).mean()
df['DGS10_roll_mean'] = df['DGS10'].rolling(window=window).mean()
df['sp500_roll_std'] = df['sp500'].rolling(window=window).std()
df['DGS10_roll_std'] = df['DGS10'].rolling(window=window).std()
df['corr'] = df['sp500'].rolling(window=window).corr(df['DGS10'])

# もう一方の変数の過去1日分のデータ（外生変数用）
df['DGS10_lag1'] = df['DGS10'].shift(1)
df['sp500_lag1'] = df['sp500'].shift(1)

# 欠損値の削除
df_features = df.dropna().copy()
print(f"有効なデータ件数: {len(df_features)}件")

features = df_features[['sp500_roll_mean', 'DGS10_roll_mean', 'sp500_roll_std', 'DGS10_roll_std', 'corr']]



## 2. GMMを用いた4つの状態（レジーム）へのクラスタリング

### 理論と計算手法（数式）
GMM（Gaussian Mixture Model: 混合ガウスモデル）は、データが複数の正規分布の集まり（混合分布）から生成されていると仮定し、それぞれのデータがどの分布から生成されたか（どの状態に属するか）を確率的に推定する手法です。ここでは $K=4$ として市場を4つの異なるレジームに分類しています。

**1. モデルの確率密度関数**
特徴量ベクトル $x$ が与えられたときの全体の確率分布 $P(x)$ は、各レジームの正規分布の線形結合として表されます。
$$
P(x) = \sum_{k=1}^K \pi_k \mathcal{N}(x \mid \mu_k, \Sigma_k)
$$

**2. EMアルゴリズムによるパラメータ推定**
未知のパラメータ $\theta = \{\pi_k, \mu_k, \Sigma_k\}$ は、尤度が最大になるように **EMアルゴリズム** によって反復計算で推定されます。

**3. レジームの判定**
EMアルゴリズムが収束した後、各時点 $t$ における市場のレジーム $S_t$ は、最も事後確率が高い分布として割り当てられます（ハードクラスタリング）。
$$
S_t = \arg\max_k \gamma(z_{tk})
$$
これにより、「平穏な上昇相場」「荒れ相場」などが自動的にグループ化されます。


In [ ]:
gmm = GaussianMixture(n_components=4, random_state=42, n_init=10)
df_features['Regime'] = gmm.fit_predict(features)

print("各レジームのデータ件数:")
print(df_features['Regime'].value_counts().sort_index())

# 状態の可視化
plt.figure(figsize=(15, 6))
plt.plot(df_features.index, df_features['sp500'], color='black', alpha=0.3, label='S&P 500 Return')
colors = ['red', 'blue', 'green', 'orange']
for i in range(4):
    mask = df_features['Regime'] == i
    plt.scatter(df_features.index[mask], df_features['sp500'][mask], color=colors[i], label=f'Regime {i}', s=10)
plt.title('S&P 500 Returns by GMM Clustered Regime')
plt.legend()
plt.show()



## 3. 状態遷移確率行列の計算

### 理論
市場のレジームは、ランダムに切り替わるわけではありません。「荒れ相場」の次の日は「荒れ相場」になりやすいといった**持続性（マルコフ性）**があります。
推移確率行列（Transition Matrix）は、「現在の状態が $i$ のとき、次の日に状態が $j$ になる確率 $P(S_{t+1}=j | S_t=i)$」を行列にしたものです。
対角成分（[0,0]や[1,1]など）が大きいほど、その状態が持続しやすいことを意味します。


In [ ]:
transition_matrix = np.zeros((4, 4))
regimes = df_features['Regime'].values
for t in range(1, len(regimes)):
    from_state = regimes[t-1]
    to_state = regimes[t]
    transition_matrix[from_state, to_state] += 1

# 行ごとに正規化して確率に変換
transition_matrix = transition_matrix / transition_matrix.sum(axis=1, keepdims=True)

print("【推移確率行列 (Transition Matrix)】")
for i in range(4):
    print(f"状態 {i} からの遷移確率: [0]:{transition_matrix[i,0]:.3f}, [1]:{transition_matrix[i,1]:.3f}, [2]:{transition_matrix[i,2]:.3f}, [3]:{transition_matrix[i,3]:.3f}")



## 4. 状態ごとのARIMAモデリング（期待値の計算）

### 理論
時系列データが全体を通して同じ法則に従う（定常的である）と仮定すると、構造変化を見落としてしまいます。そこで、レジームごとに別々のARIMAモデルを推定します。
* **ARIMA (2,1,1)**: 階差(d=1)を取りつつ、過去2日分の自己回帰(AR=2)と過去1日分の移動平均(MA=1)を考慮するモデル。
* **exog (外生変数)**: SPXのモデルにはDGS10の過去1日分のデータを、DGS10のモデルにはSPXの過去1日分のデータを組み込む（ARIMAX）。

ここでは、シミュレーションのベースラインとなる「条件付き期待値（予測値）」を計算し、全期間の「予測誤差（残差）」を繋ぎ合わせます。


In [ ]:
# モデルの保存用辞書
models_sp500 = {}
models_dgs = {}
df_features['sp500_pred'] = np.nan
df_features['DGS10_pred'] = np.nan

for i in range(4):
    print(f"\n--- レジーム {i} のモデルフィッティング ---")
    mask = df_features['Regime'] == i
    
    # 1. SPX の ARIMA モデル (exog=DGS10_lag1)
    y_sp500 = df_features['sp500'].copy()
    y_sp500[~mask] = np.nan
    exog_dgs = df_features['DGS10_lag1']
    
    try:
        model_sp500 = ARIMA(endog=y_sp500, exog=exog_dgs, order=(2, 1, 1))
        res_sp500 = model_sp500.fit()
        models_sp500[i] = res_sp500
        df_features.loc[mask, 'sp500_pred'] = res_sp500.predict()[mask]
        print(f"【SPX 推定パラメータ】\n{res_sp500.params}")
    except Exception as e:
        print(e)

    # 2. DGS10 の ARIMA モデル (exog=sp500_lag1)
    y_dgs = df_features['DGS10'].copy()
    y_dgs[~mask] = np.nan
    exog_sp500 = df_features['sp500_lag1']
    
    try:
        model_dgs = ARIMA(endog=y_dgs, exog=exog_sp500, order=(2, 1, 1))
        res_dgs = model_dgs.fit()
        models_dgs[i] = res_dgs
        df_features.loc[mask, 'DGS10_pred'] = res_dgs.predict()[mask]
        print(f"\n【DGS10 推定パラメータ】\n{res_dgs.params}")
    except Exception as e:
        print(e)

# 残差（予測誤差）を全期間連続データとして計算
df_features['resid_sp500'] = df_features['sp500'] - df_features['sp500_pred']
df_features['resid_dgs'] = df_features['DGS10'] - df_features['DGS10_pred']
df_features['resid_sp500'].fillna(0, inplace=True)
df_features['resid_dgs'].fillna(0, inplace=True)



### レジームごとの予測誤差（残差）の分布比較
各レジームにおけるARIMAモデルの予測誤差（実際の値 - 予測値）の分布を、ヒストグラムをベースにした折れ線グラフ（度数多角形）で一括比較します。
全体に対する割合（密度）で描画しているため、データ件数が異なるレジーム間でも、予測のばらつき（ボラティリティ）の形状を直接比較できます。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = ['red', 'blue', 'green', 'orange']

for i in range(4):
    mask = df_features['Regime'] == i
    res_sp = df_features.loc[mask, 'resid_sp500'].dropna().values
    res_dgs = df_features.loc[mask, 'resid_dgs'].dropna().values
    
    if len(res_sp) > 1:
        # SPXのヒストグラムの階級値（ビンの中央）と割合を計算して折れ線グラフを描画
        counts_sp, bins_sp = np.histogram(res_sp, bins=30, density=True)
        bin_centers_sp = 0.5 * (bins_sp[1:] + bins_sp[:-1])
        axes[0].plot(bin_centers_sp, counts_sp, marker='o', markersize=4, color=colors[i], label=f'Regime {i}', linewidth=1.5, alpha=0.8)
        
    if len(res_dgs) > 1:
        # DGS10のヒストグラムの階級値と割合を計算して折れ線グラフを描画
        counts_dgs, bins_dgs = np.histogram(res_dgs, bins=30, density=True)
        bin_centers_dgs = 0.5 * (bins_dgs[1:] + bins_dgs[:-1])
        axes[1].plot(bin_centers_dgs, counts_dgs, marker='o', markersize=4, color=colors[i], label=f'Regime {i}', linewidth=1.5, alpha=0.8)

axes[0].set_title('SPX Prediction Error Distribution (Proportion Line Graph)')
axes[0].set_xlabel('Prediction Error (Actual - Predicted)')
axes[0].set_ylabel('Proportion (Density)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('DGS10 Prediction Error Distribution (Proportion Line Graph)')
axes[1].set_xlabel('Prediction Error (Actual - Predicted)')
axes[1].set_ylabel('Proportion (Density)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



## 5. レジーム別共分散行列の計算とシミュレーション生成

### 理論: レジームごとの多変量正規分布によるシミュレーション

極端な動的相関（過剰なクラスタリング）を防ぐため、レジームごとに「予測誤差の分散共分散行列（固定相関）」を計算し、各時点での現在のレジームに対応する多変量正規分布からショックをサンプリングします。

$$
\begin{pmatrix} y_{sp, t} \\ y_{dgs, t} \end{pmatrix}_{sim} = \begin{pmatrix} \hat{y}_{sp, t} \\ \hat{y}_{dgs, t} \end{pmatrix} + \epsilon_t
$$
$$
\epsilon_t \sim \mathcal{N}(0, \Sigma_{S_t})
$$
ここで $\Sigma_{S_t}$ は、時点 $t$ でのレジーム $S_t$ におけるSPXとDGS10の残差の共分散行列です。これにより、平穏な時期と荒れ相場の違いは表現しつつ、極端な相関のスパイクを持たない安定したシミュレーションデータが生成されます。


In [ ]:
# レジームごとの共分散行列を計算
regime_cov_matrices = {}

for i in range(4):
    mask = df_features['Regime'] == i
    res_sp = df_features.loc[mask, 'resid_sp500'].values
    res_dgs = df_features.loc[mask, 'resid_dgs'].values
    
    # NaNを除外して計算
    valid_mask = ~np.isnan(res_sp) & ~np.isnan(res_dgs)
    if valid_mask.sum() > 1:
        cov_matrix = np.cov(res_sp[valid_mask], res_dgs[valid_mask])
        regime_cov_matrices[i] = cov_matrix
        print(f"\n【Regime {i} 共分散行列】")
        print(cov_matrix)
        # 相関係数も表示
        corr = cov_matrix[0, 1] / np.sqrt(cov_matrix[0, 0] * cov_matrix[1, 1])
        print(f"相関係数: {corr:.4f}")
    else:
        # データが足りない場合は単位行列を使用
        regime_cov_matrices[i] = np.eye(2) * 1e-6

# シミュレーションの実行
df_features['sp500_simulated'] = np.nan
df_features['DGS10_simulated'] = np.nan

shocks_sp = np.zeros(len(df_features))
shocks_dgs = np.zeros(len(df_features))

regimes = df_features['Regime'].values

np.random.seed(42) # 再現性のためのシード

for t in range(len(df_features)):
    sim_regime = regimes[t]
    cov_matrix = regime_cov_matrices[sim_regime]
    
    # 2変量正規分布からショックをサンプリング
    shock = np.random.multivariate_normal([0, 0], cov_matrix)
    
    shocks_sp[t] = shock[0]
    shocks_dgs[t] = shock[1]

# ARIMAの期待値にショックを足す
df_features['sp500_simulated'] = df_features['sp500_pred'] + shocks_sp
df_features['DGS10_simulated'] = df_features['DGS10_pred'] + shocks_dgs

print("\nレジーム別の固定相関（共分散行列）を反映したシミュレーションデータを生成しました！")

round_cols = ['sp500_pred', 'sp500_simulated', 'DGS10_pred', 'DGS10_simulated']
df_features[round_cols] = df_features[round_cols].round(6)

output_df = df_features[['Regime', 'sp500', 'sp500_pred', 'sp500_simulated', 'DGS10', 'DGS10_pred', 'DGS10_simulated']]
print("【直近10日間の実際の値と予測値・シミュレーション値】")
display(output_df.tail(10))

# 実際の値と予測値・シミュレーション値のプロット
plt.figure(figsize=(15, 6))
plt.plot(output_df.index[-100:], output_df['sp500'][-100:], label='Actual SPX', color='black', alpha=0.5)
plt.plot(output_df.index[-100:], output_df['sp500_pred'][-100:], label='Predicted SPX (Mean)', color='blue', linestyle='--')
plt.plot(output_df.index[-100:], output_df['sp500_simulated'][-100:], label='Simulated SPX (with regime shocks)', color='red', alpha=0.7)
plt.title('SPX Actual vs Predicted vs Simulated (Last 100 observations)')
plt.xticks(rotation=45)
plt.legend()
plt.show()



## 6. 全期間での累積リターンの比較

初期値を $1$ として、実際のSPXリターンとモデルの予測リターン（`sp500_pred`）を累積（複利計算）して比較します。
これにより、モデルの予測通りに運用した場合の長期的なパフォーマンスと、実際の市場のパフォーマンスの乖離を視覚的に評価できます。


In [ ]:
# 欠損値（NaN）を0で埋めてから累積積（cumprod）を計算
df_features['cum_sp500'] = (1 + df_features['sp500'].fillna(0)).cumprod()
df_features['cum_sp500_pred'] = (1 + df_features['sp500_pred'].fillna(0)).cumprod()
df_features['cum_sp500_sim'] = (1 + df_features['sp500_simulated'].fillna(0)).cumprod()

plt.figure(figsize=(15, 6))
plt.plot(df_features.index, df_features['cum_sp500'], label='Actual SPX Cumulative Return', color='black')
plt.plot(df_features.index, df_features['cum_sp500_pred'], label='Predicted SPX Cumulative Return', color='blue', linestyle='--')
plt.plot(df_features.index, df_features['cum_sp500_sim'], label='Simulated SPX Cumulative Return', color='red', alpha=0.7)
plt.title('Cumulative Returns: Actual vs Predicted vs Simulated (Initial Value = 1)')
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



## 7. 実データとモデル生成データ（予測・シミュレーション）の相関比較

モデルが、S&P 500 と米国10年国債利回り (DGS10) の「2資産間の関係性（相関）」をどれだけ上手く再現できているかを検証します。
今回はレジームごとの固定相関を使用しているため、極端なスパイクのない安定した連動性がシミュレーションに反映されています。


In [ ]:
# 1年ごとのビンを作成
df_features['1Y_Label'] = df_features.index.year.astype(str)

# 1年ごとに相関を計算
corr_act = df_features.groupby('1Y_Label').apply(lambda g: g['sp500'].corr(g['DGS10']))
corr_pred = df_features.groupby('1Y_Label').apply(lambda g: g['sp500_pred'].corr(g['DGS10_pred']))
corr_sim = df_features.groupby('1Y_Label').apply(lambda g: g['sp500_simulated'].corr(g['DGS10_simulated']))

corr_1y = pd.DataFrame({'Actual': corr_act, 'Predicted': corr_pred, 'Simulated': corr_sim})

# 折れ線グラフで視覚化
plt.figure(figsize=(20, 6))
plt.plot(corr_1y.index, corr_1y['Actual'], marker='o', label='Actual Data', color='black', linewidth=1.5, markersize=4)
plt.plot(corr_1y.index, corr_1y['Predicted'], marker='x', label='Predicted (Mean)', color='blue', linestyle='--', linewidth=1.5, markersize=4)
plt.plot(corr_1y.index, corr_1y['Simulated'], marker='s', label='Simulated (with regime shocks)', color='red', alpha=0.7, linewidth=1.5, markersize=4)

plt.axhline(0, color='gray', linewidth=1, linestyle='--')
plt.title('1-Year Rolling Correlation between SPX and DGS10 (Actual vs Predicted vs Simulated)')
plt.xlabel('Year')
plt.ylabel('Correlation Coefficient')
plt.xticks(rotation=90, fontsize=8)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 相関係数のボラティリティ比較
vol_act = corr_1y['Actual'].std()
vol_pred = corr_1y['Predicted'].std()
vol_sim = corr_1y['Simulated'].std()

print("\n【相関係数のボラティリティ（標準偏差）の比較】")

# ボラティリティを棒グラフで視覚化
plt.figure(figsize=(8, 5))
labels_vol = ['Actual Data', 'Predicted (Mean)', 'Simulated (with regime shocks)']
values_vol = [vol_act, vol_pred, vol_sim]
colors_vol = ['gray', 'blue', 'red']

plt.bar(labels_vol, values_vol, color=colors_vol, alpha=0.7)
plt.title('Volatility (Standard Deviation) of 1-Year Rolling Correlation')
plt.ylabel('Volatility of Correlation')
for i, v in enumerate(values_vol):
    if pd.notna(v):
        plt.text(i, v + (max([val for val in values_vol if pd.notna(val)] + [0])*0.02), f"{v:.4f}", ha='center', fontweight='bold')
plt.ylim(0, max([val for val in values_vol if pd.notna(val)] + [0.1]) * 1.2)
plt.show()
